In [ ]:
import pandas as pd

from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split

In [2]:
data = pd.read_csv("reviews.csv")

print("Dataset Loaded Successfully\n")

print(data.head())

Dataset Loaded Successfully

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [3]:
data = data.dropna()

print("Missing Values Removed")

Missing Values Removed


In [4]:
# Input Reviews
X = data["review"]

# Output Labels
y = data["sentiment"]

print("Input and Output Separated")

Input and Output Separated


In [5]:
# positive = 1
# negative = 0

y = y.map({
    'positive': 1,
    'negative': 0
})

print("Labels Converted Successfully")

Labels Converted Successfully


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data:", len(X_train))
print("Testing Data:", len(X_test))

Training Data: 40000
Testing Data: 10000


In [7]:
tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)

X_test = tokenizer.texts_to_sequences(X_test)

print("Text Converted into Numbers")

Text Converted into Numbers


In [8]:
X_train = pad_sequences(X_train, maxlen=200)

X_test = pad_sequences(X_test, maxlen=200)

print("Padding Completed")

Padding Completed


In [9]:
model = keras.Sequential([

    keras.Input(shape=(200,)),

    # Embedding Layer
    keras.layers.Embedding(10000, 128),

    # Flatten Layer
    keras.layers.Flatten(),

    # Hidden Layers
    keras.layers.Dense(64, activation='relu'),

    keras.layers.Dense(32, activation='relu'),

    # Output Layer
    keras.layers.Dense(1, activation='sigmoid')
])

print("Model Created Successfully")

Model Created Successfully


In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,638,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,920,577 (11.14 MB)

 Trainable params: 2,920,577 (11.14 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model Compiled Successfully")

Model Compiled Successfully


In [12]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 36s 33ms/step - accuracy: 0.8207 - loss: 0.3781 - val_accuracy: 0.8748 - val_loss: 0.2861
Epoch 2/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 31s 31ms/step - accuracy: 0.9709 - loss: 0.0858 - val_accuracy: 0.8519 - val_loss: 0.4372
Epoch 3/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 31s 31ms/step - accuracy: 0.9936 - loss: 0.0184 - val_accuracy: 0.8379 - val_loss: 0.7344
Epoch 4/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 42s 32ms/step - accuracy: 0.9934 - loss: 0.0182 - val_accuracy: 0.8480 - val_loss: 0.7290
Epoch 5/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 31s 31ms/step - accuracy: 0.9948 - loss: 0.0146 - val_accuracy: 0.8576 - val_loss: 0.6783


In [13]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)

print("Test Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8580 - loss: 0.6686
Test Loss: 0.6685550808906555
Test Accuracy: 0.8579999804496765


In [14]:
predictions = model.predict(X_test[:8])

for i, pred in enumerate(predictions):

    if pred[0] > 0.5:
        print("Review", i+1, ": Positive Review")
    else:
        print("Review", i+1, ": Negative Review")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
Review 1 : Negative Review
Review 2 : Positive Review
Review 3 : Negative Review
Review 4 : Positive Review
Review 5 : Negative Review
Review 6 : Positive Review
Review 7 : Positive Review
Review 8 : Negative Review
